# P131 — Una marca de agua para modelos de lenguaje grandes

## 1. Título y paper

**Paper:** *A Watermark for Large Language Models*  
**Autoría:** John Kirchenbauer, Jonas Geiping, Yuxin Wen, Jonathan Katz, Ian Miers, Tom Goldstein  
**Año y venue:** 2023 · ICML 2023 · arXiv:2301.10226  
**Nivel:** L2 · **Motor:** `marcas_de_agua`  
**Ficha completa:** [`P131_marcas_de_agua`](../../papers/foundational/P131_marcas_de_agua/README.md)

**Hito:** Deja una firma estadística verificable en el texto generado sesgando qué tokens se eligen, sin degradar apreciablemente la calidad ni necesitar el modelo para detectarla.

- [arXiv:2301.10226](https://arxiv.org/abs/2301.10226)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Distinguir texto generado de texto humano se intentaba con clasificadores entrenados a posteriori, que fallan, envejecen con cada modelo nuevo y producen falsos positivos con consecuencias reales sobre personas.
2. Ejecutar una implementación mínima de la propuesta: Partir el vocabulario en cada paso en una lista «verde» y otra «roja», determinadas por un hash del token anterior, y sesgar la generación hacia la verde. Un texto marcado tiene una proporción de verdes anómala, y una prueba estadística la detecta sin acceso al modelo.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P10
- P130


## 4. Intuición

En vez de intentar reconocer texto generado a posteriori —que falla—, dejar la marca **al generar**: sesgar qué tokens se eligen deja una firma que una prueba estadística encuentra.


## 5. Concepto mínimo

```text
En cada paso: hash(token anterior) → lista VERDE (γ del vocabulario)
              sesgar la elección hacia la verde con δ

Detección:  z = (verdes − γn) / √(n·γ(1−γ))       sin acceso al modelo
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('marcas_de_agua', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Qué puntuación z da un texto marcado y uno humano?
2. ¿Cuánto texto hace falta?
3. ¿Qué pasa si se reescribe parte?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('marcas_de_agua', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('marcas_de_agua', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Marcado: **z = 4,83**. Humano: **z = −0,13**. Con umbral 4,0 se detectan **25 de 30** y hay **0 falsos positivos**. Pero con **25 tokens** solo se detecta **1 de 30**, y reescribiendo el **30 %** de los tokens la detección cae a **2 de 30**.


## 10. Comentario pedagógico

Las dos últimas cifras son el límite práctico. **Un tuit no se puede marcar** —no hay suficiente texto para que la estadística diga nada— y **parafrasear es un ataque barato**. La marca de agua sirve para texto largo sin editar, que es un caso de uso real pero mucho más estrecho de lo que sugiere el titular.


## 11. Error o anti-patrón deliberado

Anti-patrón: presentar la marca de agua como solución a la desinformación.


In [ ]:
print('La marca identifica al GENERADOR, no al autor ni la intencion.')
print('No dice si el texto es cierto, ni si quien lo publico sabia que era generado.')
print('Y solo funciona si el generador coopera: un modelo abierto no la lleva.')

## 12. Corrección

Los tres regímenes: detección, longitud y robustez.


In [ ]:
r = run_paper_lab('marcas_de_agua', seed=3)['result']
print('z marcado:', r['z_media_texto_marcado'], '| z humano:', r['z_media_texto_no_marcado'])
print('longitud:')
for f in r['efecto_de_la_longitud']:
    print('  ', f)
print('robustez:')
for f in r['robustez_a_la_reescritura']:
    print('  ', f)

## 13. Desafío guiado

Explica qué garantiza exactamente una marca de agua detectada y qué NO garantiza, y por qué esa distinción importa en un procedimiento disciplinario.


In [ ]:
r = run_paper_lab('marcas_de_agua', seed=3)['result']
show(r)

## 14. Desafío autónomo

Define para un caso tuyo el umbral de detección que usarías y calcula qué tasa de falsos positivos aceptas. Razona qué le pasa a una persona acusada por un falso positivo.


## 15. Evidencia de aprendizaje

Guarda el umbral, la tasa de falsos positivos y tu justificación.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P131_marcas_de_agua/README.md) · evaluación formal: [`assessments/papers/P131_marcas_de_agua.md`](../../assessments/papers/P131_marcas_de_agua.md)


## 16. Cierre

Saber qué se generó importa por una razón concreta que cierra la ruta: lo generado vuelve al corpus.


## 17. Conexión con el siguiente hito

- P133

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
